# CentaurDrug — Production-Oriented ADMET Prototype: AqSolDB with XGBoost

This notebook is not a toy baseline. It is a **production-oriented prototype** for the first ADMET model:

```text
SMILES → aqueous solubility prediction
```

Dataset:

```text
TDC ADME: Solubility_AqSolDB
```

Model:

```text
XGBoost regressor
```

Improvements over the previous notebook:

- explicit SMILES validation contract
- consistent rejection path for invalid molecules
- Morgan fingerprint + MACCS keys + RDKit physicochemical descriptors
- scaffold-aware splitting logic
- scaffold-group cross-validation
- hyperparameter search
- true train / early-stopping / validation / test separation
- early stopping using a separate tuning holdout
- reproducibility seeds
- structured logging
- optional MLflow tracking
- production-style artifact saving
- inference function with clear output contract

Run from the project root:

```bash
cd ~/Desktop/projects/centaurdrug
uv add pytdc rdkit xgboost scikit-learn pandas numpy joblib pyyaml mlflow jupyter
uv run jupyter lab
```

If you do not want MLflow yet, this notebook still works without starting an MLflow server.

## 1. Imports, configuration, reproducibility

In [6]:
%pip install pytdc rdkit scikit-learn xgboost pandas numpy joblib matplotlib jupyter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pytdc to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 16.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 120.8 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py)

In [1]:
from __future__ import annotations

import json
import logging
import os
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

from tdc.single_pred import ADME

from rdkit import Chem, DataStructs
from rdkit import RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold, ParameterSampler, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except Exception:
    mlflow = None
    MLFLOW_AVAILABLE = False

RDLogger.DisableLog("rdApp.*")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "models" / "admet_aqsol_xgboost_prototype"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "Solubility_AqSolDB"
TARGET_COL = "Y"
SMILES_COL = "Drug"

USE_MLFLOW = MLFLOW_AVAILABLE
MLFLOW_EXPERIMENT_NAME = "centaurdrug-admet-aqsol-xgboost"

logger = logging.getLogger("centaurdrug.admet.prototype")
logger.setLevel(logging.INFO)
logger.handlers.clear()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter(
    fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
))
logger.addHandler(handler)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Artifact dir: %s", ARTIFACT_DIR)
logger.info("MLflow available: %s", MLFLOW_AVAILABLE)

2026-05-28 20:30:09 | INFO | centaurdrug.admet.prototype | Project root: /home/aliaqil/Desktop/projects/centaurdrug/prototypes
2026-05-28 20:30:09 | INFO | centaurdrug.admet.prototype | Artifact dir: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype
2026-05-28 20:30:09 | INFO | centaurdrug.admet.prototype | MLflow available: True


## 2. Input validation contract

Production rule:

```text
Invalid SMILES are not silently ignored.
They produce a structured rejection record.
```

During training we keep valid rows and save a rejection report.

During inference we return a clear response:

```json
{
  "status": "rejected",
  "reason": "invalid_smiles"
}
```


In [2]:
@dataclass
class MoleculeValidationResult:
    original_smiles: str
    is_valid: bool
    canonical_smiles: Optional[str] = None
    rejection_reason: Optional[str] = None


def validate_smiles(smiles: Any) -> MoleculeValidationResult:
    """Validate and canonicalize one SMILES string."""
    if smiles is None:
        return MoleculeValidationResult(str(smiles), False, rejection_reason="missing_smiles")

    smiles_str = str(smiles).strip()
    if not smiles_str:
        return MoleculeValidationResult(str(smiles), False, rejection_reason="empty_smiles")

    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return MoleculeValidationResult(smiles_str, False, rejection_reason="invalid_smiles")

    canonical = Chem.MolToSmiles(mol, canonical=True)
    return MoleculeValidationResult(smiles_str, True, canonical_smiles=canonical)


def validate_dataframe(df: pd.DataFrame, smiles_col: str = SMILES_COL) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return valid dataframe and rejected dataframe."""
    records = []
    for idx, smiles in df[smiles_col].items():
        result = validate_smiles(smiles)
        rec = asdict(result)
        rec["row_index"] = idx
        records.append(rec)

    validation_df = pd.DataFrame(records).set_index("row_index")
    merged = df.join(validation_df)

    valid_df = merged[merged["is_valid"]].copy()
    rejected_df = merged[~merged["is_valid"]].copy()

    valid_df[smiles_col] = valid_df["canonical_smiles"]

    return valid_df, rejected_df


for s in ["CCO", "", "not_a_smiles", None]:
    print(validate_smiles(s))

MoleculeValidationResult(original_smiles='CCO', is_valid=True, canonical_smiles='CCO', rejection_reason=None)
MoleculeValidationResult(original_smiles='', is_valid=False, canonical_smiles=None, rejection_reason='empty_smiles')
MoleculeValidationResult(original_smiles='not_a_smiles', is_valid=False, canonical_smiles=None, rejection_reason='invalid_smiles')
MoleculeValidationResult(original_smiles='None', is_valid=False, canonical_smiles=None, rejection_reason='missing_smiles')


## 3. Load TDC AqSolDB dataset

In [3]:
def load_tdc_adme_dataset(name: str = DATASET_NAME) -> pd.DataFrame:
    """Load full TDC dataset as one dataframe. We create our own scaffold-aware splits."""
    data = ADME(name=name)
    full_df = data.get_data()
    full_df = full_df[[SMILES_COL, TARGET_COL]].copy()
    return full_df


raw_df = load_tdc_adme_dataset(DATASET_NAME)
logger.info("Raw dataset shape: %s", raw_df.shape)
display(raw_df.head())
display(raw_df[TARGET_COL].describe())

Downloading...
100%|██████████| 853k/853k [00:00<00:00, 1.06MiB/s]
Loading...
Done!
2026-05-28 20:31:10 | INFO | centaurdrug.admet.prototype | Raw dataset shape: (9982, 2)


,Drug,Y
0,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],-3.616127
1,O=C1Nc2cccc3cccc1c23,-3.254767
2,O=Cc1ccc(Cl)cc1,-2.177078
3,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,-3.924409
4,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,-4.662065


count    9982.000000
mean       -2.889909
std         2.368154
min       -13.171900
25%        -4.326325
50%        -2.618173
75%        -1.209735
max         2.137682
Name: Y, dtype: float64

## 4. Validate data and save rejection report

In [4]:
valid_df, rejected_df = validate_dataframe(raw_df, SMILES_COL)

logger.info("Valid molecules: %d", len(valid_df))
logger.info("Rejected molecules: %d", len(rejected_df))

rejection_path = ARTIFACT_DIR / "training_rejections.csv"
rejected_df.to_csv(rejection_path, index=False)
logger.info("Saved rejection report: %s", rejection_path)

display(valid_df.head())
display(rejected_df.head())

2026-05-28 20:31:21 | INFO | centaurdrug.admet.prototype | Valid molecules: 9980
2026-05-28 20:31:21 | INFO | centaurdrug.admet.prototype | Rejected molecules: 2
2026-05-28 20:31:21 | INFO | centaurdrug.admet.prototype | Saved rejection report: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/training_rejections.csv


,Drug,Y,original_smiles,is_valid,canonical_smiles,rejection_reason
0,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],-3.616127,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],True,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],None
1,O=C1Nc2cccc3cccc1c23,-3.254767,O=C1Nc2cccc3cccc1c23,True,O=C1Nc2cccc3cccc1c23,None
2,O=Cc1ccc(Cl)cc1,-2.177078,O=Cc1ccc(Cl)cc1,True,O=Cc1ccc(Cl)cc1,None
3,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,-3.924409,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,True,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,None
4,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,-4.662065,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,True,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,None


,Drug,Y,original_smiles,is_valid,canonical_smiles,rejection_reason
4792,CC1=CC=C[NH+2]([O-])[CH-]1,0.9621,CC1=CC=C[NH+2]([O-])[CH-]1,False,None,invalid_smiles
5043,O=C(O)C1=C[NH+2]([O-])[CH-]C=C1,-1.2983,O=C(O)C1=C[NH+2]([O-])[CH-]C=C1,False,None,invalid_smiles


## 5. Scaffold generation

Why this matters:

Random molecular splits can make the model look too good because similar analogues appear in both train and test.

Scaffold splitting is harder and more realistic:

```text
same Bemis–Murcko scaffold → same group
```


In [5]:
def compute_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "invalid"
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    if not scaffold:
        # acyclic molecules often have empty Murcko scaffold; group by canonical smiles fallback
        return f"acyclic::{Chem.MolToSmiles(mol, canonical=True)}"
    return scaffold


valid_df["scaffold"] = valid_df[SMILES_COL].apply(compute_scaffold)

logger.info("Unique scaffolds: %d", valid_df["scaffold"].nunique())
display(valid_df[[SMILES_COL, TARGET_COL, "scaffold"]].head())
display(valid_df["scaffold"].value_counts().head(10))

2026-05-28 20:31:41 | INFO | centaurdrug.admet.prototype | Unique scaffolds: 4886


,Drug,Y,scaffold
0,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-],-3.616127,acyclic::CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-]
1,O=C1Nc2cccc3cccc1c23,-3.254767,O=C1Nc2cccc3cccc1c23
2,O=Cc1ccc(Cl)cc1,-2.177078,c1ccccc1
3,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...,-3.924409,c1ccc(Cc2cccc(Cc3ccccc3)c2)cc1.c1ccc(Cc2cccc(C...
4,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...,-4.662065,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...


scaffold
c1ccccc1                        1753
c1ccc(-c2ccccc2)cc1              219
c1ccc2ccccc2c1                   152
c1ccc(Oc2ccccc2)cc1              139
C1CCCCC1                         117
c1ccncc1                         102
c1ccc(Cc2ccccc2)cc1               73
c1ccc(N=Nc2ccccc2)cc1             61
O=C1C=C2CCC3C4CCCC4CCC3C2CC1      54
O=c1cc[nH]c(=O)[nH]1              52
Name: count, dtype: int64

## 6. Scaffold-aware train / validation / test split

We create four logical sets:

```text
train_core       → used for model fitting
early_stop       → used only for early stopping
validation       → used for hyperparameter selection / model comparison
test             → final untouched estimate
```

This fixes the methodological issue where the same validation set is used both for early stopping and reported validation metrics.

In [6]:
def scaffold_ordered_split(
    df: pd.DataFrame,
    scaffold_col: str = "scaffold",
    train_frac: float = 0.70,
    early_stop_frac: float = 0.10,
    valid_frac: float = 0.10,
    test_frac: float = 0.10,
    seed: int = RANDOM_SEED,
) -> Dict[str, pd.DataFrame]:
    """Deterministic scaffold split. Groups are shuffled, then assigned by molecule counts."""
    assert abs(train_frac + early_stop_frac + valid_frac + test_frac - 1.0) < 1e-8

    groups = list(df.groupby(scaffold_col))
    rng = random.Random(seed)
    rng.shuffle(groups)

    n_total = len(df)
    targets = {
        "train_core": train_frac * n_total,
        "early_stop": early_stop_frac * n_total,
        "validation": valid_frac * n_total,
        "test": test_frac * n_total,
    }

    buckets = {k: [] for k in targets}
    counts = {k: 0 for k in targets}

    for scaffold, group in groups:
        # assign to bucket with largest remaining capacity
        remaining = {k: targets[k] - counts[k] for k in targets}
        bucket = max(remaining, key=remaining.get)
        buckets[bucket].append(group)
        counts[bucket] += len(group)

    return {
        k: pd.concat(v).sample(frac=1.0, random_state=seed).reset_index(drop=True)
        for k, v in buckets.items()
        if v
    }


splits = scaffold_ordered_split(valid_df)

for name, split_df in splits.items():
    logger.info("%s shape: %s | unique scaffolds: %d", name, split_df.shape, split_df["scaffold"].nunique())
    print(name, split_df.shape)

split_report = pd.DataFrame({
    "split": list(splits.keys()),
    "n_molecules": [len(v) for v in splits.values()],
    "n_scaffolds": [v["scaffold"].nunique() for v in splits.values()],
    "target_mean": [v[TARGET_COL].mean() for v in splits.values()],
    "target_std": [v[TARGET_COL].std() for v in splits.values()],
})
display(split_report)

split_report.to_csv(ARTIFACT_DIR / "split_report.csv", index=False)

2026-05-28 20:32:00 | INFO | centaurdrug.admet.prototype | train_core shape: (7964, 7) | unique scaffolds: 3748
2026-05-28 20:32:00 | INFO | centaurdrug.admet.prototype | early_stop shape: (669, 7) | unique scaffolds: 505
2026-05-28 20:32:00 | INFO | centaurdrug.admet.prototype | validation shape: (668, 7) | unique scaffolds: 433
2026-05-28 20:32:00 | INFO | centaurdrug.admet.prototype | test shape: (679, 7) | unique scaffolds: 200


train_core (7964, 7)
early_stop (669, 7)
validation (668, 7)
test (679, 7)


,split,n_molecules,n_scaffolds,target_mean,target_std
0,train_core,7964,3748,-2.731004,2.220747
1,early_stop,669,505,-2.771031,2.487534
2,validation,668,433,-2.773407,2.362529
3,test,679,200,-4.993465,2.875449


## 7. Feature engineering: Morgan + MACCS + RDKit descriptors

This is still cheap classical ML, but much stronger than Morgan alone.

Feature vector:

```text
[Morgan fingerprint | MACCS keys | RDKit descriptors]
```

RDKit descriptors are scaled; binary fingerprints are kept as binary features.

In [7]:
RDKit_DESCRIPTOR_FUNCS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Descriptors.MolLogP),
    ("TPSA", Descriptors.TPSA),
    ("NumHDonors", Descriptors.NumHDonors),
    ("NumHAcceptors", Descriptors.NumHAcceptors),
    ("NumRotatableBonds", Descriptors.NumRotatableBonds),
    ("RingCount", Descriptors.RingCount),
    ("HeavyAtomCount", Descriptors.HeavyAtomCount),
    ("FractionCSP3", rdMolDescriptors.CalcFractionCSP3),
    ("NHOHCount", Descriptors.NHOHCount),
    ("NOCount", Descriptors.NOCount),
    ("NumAliphaticRings", rdMolDescriptors.CalcNumAliphaticRings),
    ("NumAromaticRings", rdMolDescriptors.CalcNumAromaticRings),
    ("NumSaturatedRings", rdMolDescriptors.CalcNumSaturatedRings),
]


class MolecularFeatureExtractor(BaseEstimator, TransformerMixin):
    """SMILES → Morgan + MACCS + selected RDKit descriptors."""

    def __init__(self, radius: int = 2, n_bits: int = 2048):
        self.radius = radius
        self.n_bits = n_bits
        self.descriptor_names_ = [name for name, _ in RDKit_DESCRIPTOR_FUNCS]

    def fit(self, smiles: List[str], y: Optional[np.ndarray] = None):
        return self

    def transform(self, smiles: List[str]) -> np.ndarray:
        features = []
        for s in smiles:
            mol = Chem.MolFromSmiles(str(s))
            if mol is None:
                raise ValueError(f"Invalid SMILES passed to featurizer: {s}")

            # Morgan fingerprint
            morgan = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.n_bits)
            morgan_arr = np.zeros((self.n_bits,), dtype=np.float32)
            DataStructs.ConvertToNumpyArray(morgan, morgan_arr)

            # MACCS keys: length 167
            maccs = MACCSkeys.GenMACCSKeys(mol)
            maccs_arr = np.zeros((maccs.GetNumBits(),), dtype=np.float32)
            DataStructs.ConvertToNumpyArray(maccs, maccs_arr)

            # Descriptors
            desc_values = []
            for _, func in RDKit_DESCRIPTOR_FUNCS:
                try:
                    value = float(func(mol))
                except Exception:
                    value = np.nan
                desc_values.append(value)
            desc_arr = np.asarray(desc_values, dtype=np.float32)

            full = np.concatenate([morgan_arr, maccs_arr, desc_arr])
            features.append(full)

        X = np.vstack(features).astype(np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        return X


feature_extractor = MolecularFeatureExtractor(radius=2, n_bits=2048)
sample_X = feature_extractor.transform(valid_df[SMILES_COL].head(3).tolist())
logger.info("Feature shape for 3 molecules: %s", sample_X.shape)

2026-05-28 20:32:20 | INFO | centaurdrug.admet.prototype | Feature shape for 3 molecules: (3, 2229)


## 8. Prepare arrays

In [8]:
train_core_df = splits["train_core"]
early_stop_df = splits["early_stop"]
validation_df = splits["validation"]
test_df = splits["test"]

X_train_smiles = train_core_df[SMILES_COL].tolist()
y_train = train_core_df[TARGET_COL].astype(float).to_numpy()

X_early_smiles = early_stop_df[SMILES_COL].tolist()
y_early = early_stop_df[TARGET_COL].astype(float).to_numpy()

X_valid_smiles = validation_df[SMILES_COL].tolist()
y_valid = validation_df[TARGET_COL].astype(float).to_numpy()

X_test_smiles = test_df[SMILES_COL].tolist()
y_test = test_df[TARGET_COL].astype(float).to_numpy()

X_train = feature_extractor.transform(X_train_smiles)
X_early = feature_extractor.transform(X_early_smiles)
X_valid = feature_extractor.transform(X_valid_smiles)
X_test = feature_extractor.transform(X_test_smiles)

# Scale all features. This is not essential for trees, but helpful when descriptor ranges differ
# and keeps the pipeline stable if we later add linear/blended models.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_early_scaled = scaler.transform(X_early)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled:", X_train_scaled.shape)
print("X_early_scaled:", X_early_scaled.shape)
print("X_valid_scaled:", X_valid_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

X_train_scaled: (7964, 2229)
X_early_scaled: (669, 2229)
X_valid_scaled: (668, 2229)
X_test_scaled : (679, 2229)


## 9. Metrics

In [9]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mse": float(mse),
        "rmse": float(np.sqrt(mse)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def print_metrics(title: str, metrics: Dict[str, float]):
    print(title)
    for k, v in metrics.items():
        print(f"  {k}: {v:.5f}")

## 10. Scaffold-group cross-validation for hyperparameter search

We tune hyperparameters using `GroupKFold` where the group is the molecular scaffold.

This is more honest than random CV for molecules.

In [10]:
param_distributions = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "reg_alpha": [0.0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
}

# Keep this small for prototype speed. Increase to 30-60 later.
N_PARAM_SAMPLES = 12
N_CV_SPLITS = 5

candidate_params = list(ParameterSampler(
    param_distributions,
    n_iter=N_PARAM_SAMPLES,
    random_state=RANDOM_SEED,
))

cv_groups = train_core_df["scaffold"].to_numpy()
group_kfold = GroupKFold(n_splits=N_CV_SPLITS)

logger.info("Hyperparameter candidates: %d", len(candidate_params))
logger.info("CV splits: %d", N_CV_SPLITS)

2026-05-28 20:33:18 | INFO | centaurdrug.admet.prototype | Hyperparameter candidates: 12
2026-05-28 20:33:18 | INFO | centaurdrug.admet.prototype | CV splits: 5


In [11]:
def build_xgb_regressor(params: Dict[str, Any], seed: int = RANDOM_SEED) -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=-1,
        tree_method="hist",
        **params,
    )


cv_results = []
start_time = time.time()

for i, params in enumerate(candidate_params, start=1):
    fold_metrics = []
    logger.info("Candidate %d/%d: %s", i, len(candidate_params), params)

    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train_scaled, y_train, groups=cv_groups), start=1):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = build_xgb_regressor(params)
        model.fit(X_tr, y_tr, verbose=False)

        pred = model.predict(X_val)
        metrics = regression_metrics(y_val, pred)
        fold_metrics.append(metrics)

    row = {"candidate": i, **params}
    for metric_name in ["mae", "mse", "rmse", "r2"]:
        values = [m[metric_name] for m in fold_metrics]
        row[f"cv_{metric_name}_mean"] = float(np.mean(values))
        row[f"cv_{metric_name}_std"] = float(np.std(values))

    cv_results.append(row)

elapsed = time.time() - start_time
logger.info("CV search completed in %.2f seconds", elapsed)

cv_results_df = pd.DataFrame(cv_results).sort_values("cv_rmse_mean")
display(cv_results_df.head(10))

cv_results_path = ARTIFACT_DIR / "cv_hyperparameter_results.csv"
cv_results_df.to_csv(cv_results_path, index=False)
logger.info("Saved CV results: %s", cv_results_path)

2026-05-28 20:33:22 | INFO | centaurdrug.admet.prototype | Candidate 1/12: {'subsample': 0.9, 'reg_lambda': 10.0, 'reg_alpha': 0.01, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.08, 'colsample_bytree': 1.0}
2026-05-28 20:34:15 | INFO | centaurdrug.admet.prototype | Candidate 2/12: {'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.01, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
2026-05-28 20:35:24 | INFO | centaurdrug.admet.prototype | Candidate 3/12: {'subsample': 0.7, 'reg_lambda': 0.5, 'reg_alpha': 1.0, 'n_estimators': 700, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
2026-05-28 20:36:54 | INFO | centaurdrug.admet.prototype | Candidate 4/12: {'subsample': 0.9, 'reg_lambda': 5.0, 'reg_alpha': 0.0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
2026-05-28 20:37:35 | INFO | centaurdrug.

,candidate,subsample,reg_lambda,reg_alpha,n_estimators,min_child_weight,max_depth,learning_rate,colsample_bytree,cv_mae_mean,cv_mae_std,cv_mse_mean,cv_mse_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std
10,11,0.9,5.0,1.00,300,1,8,0.03,0.7,0.737800,0.064033,1.115330,0.205235,1.050814,0.105451,0.766967,0.035040
8,9,0.8,1.0,0.01,700,10,8,0.01,0.6,0.744830,0.068265,1.124794,0.211439,1.055009,0.108399,0.765414,0.033891
11,12,0.9,10.0,0.10,500,3,6,0.03,0.9,0.752287,0.071620,1.147519,0.219385,1.065400,0.111540,0.761059,0.033127
4,5,1.0,10.0,0.01,700,5,6,0.05,1.0,0.755338,0.068262,1.154422,0.216808,1.068829,0.109668,0.759241,0.034627
9,10,0.9,1.0,1.00,500,3,8,0.01,0.9,0.755070,0.070272,1.155506,0.217116,1.069328,0.109742,0.759055,0.034434
1,2,1.0,5.0,0.01,500,3,5,0.05,0.6,0.763151,0.076424,1.164371,0.229972,1.072736,0.116655,0.758367,0.030480
5,6,0.7,2.0,0.01,700,1,4,0.03,1.0,0.772964,0.078772,1.186348,0.234177,1.082827,0.117622,0.753715,0.031661
7,8,0.9,1.0,0.00,700,10,5,0.01,0.8,0.783310,0.080028,1.214220,0.244646,1.095178,0.121678,0.748467,0.030293
0,1,0.9,10.0,0.01,300,3,3,0.08,1.0,0.793981,0.082893,1.243843,0.253141,1.108294,0.124605,0.742594,0.030103
2,3,0.7,0.5,1.00,700,5,3,0.01,0.6,0.837945,0.099371,1.345809,0.293841,1.151583,0.140233,0.723589,0.024440


2026-05-28 20:49:58 | INFO | centaurdrug.admet.prototype | Saved CV results: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/cv_hyperparameter_results.csv


## 11. Train final model with early stopping

Now we use:

```text
train_core → model fitting
early_stop → early stopping only
validation → honest model selection report
test → final untouched score
```

Important:

- early stopping is configured
- validation set is not used for early stopping
- test set is used once at the end

In [12]:
best_params = cv_results_df.iloc[0].to_dict()

# Remove metadata columns from row
xgb_param_keys = set(param_distributions.keys())
best_params = {k: best_params[k] for k in xgb_param_keys}

# Ensure integer params stay integers
for k in ["n_estimators", "max_depth", "min_child_weight"]:
    best_params[k] = int(best_params[k])

logger.info("Best params from CV: %s", best_params)
best_params

2026-05-28 20:50:14 | INFO | centaurdrug.admet.prototype | Best params from CV: {'subsample': 0.9, 'min_child_weight': 1, 'reg_lambda': 5.0, 'reg_alpha': 1.0, 'colsample_bytree': 0.7, 'max_depth': 8, 'learning_rate': 0.03, 'n_estimators': 300}


{'subsample': 0.9,
 'min_child_weight': 1,
 'reg_lambda': 5.0,
 'reg_alpha': 1.0,
 'colsample_bytree': 0.7,
 'max_depth': 8,
 'learning_rate': 0.03,
 'n_estimators': 300}

In [13]:
final_model = build_xgb_regressor(best_params)

# XGBoost's sklearn API changed across versions. This block supports both styles.
try:
    final_model.fit(
        X_train_scaled,
        y_train,
        eval_set=[(X_early_scaled, y_early)],
        early_stopping_rounds=50,
        verbose=False,
    )
except TypeError:
    # Newer xgboost versions may require early_stopping_rounds in constructor.
    logger.warning("fit(..., early_stopping_rounds=...) unsupported. Retrying with constructor early_stopping_rounds.")
    final_model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=50,
        **best_params,
    )
    final_model.fit(
        X_train_scaled,
        y_train,
        eval_set=[(X_early_scaled, y_early)],
        verbose=False,
    )

logger.info("Best iteration: %s", getattr(final_model, "best_iteration", None))
logger.info("Best score: %s", getattr(final_model, "best_score", None))

2026-05-28 20:50:18 | WARNING | centaurdrug.admet.prototype | fit(..., early_stopping_rounds=...) unsupported. Retrying with constructor early_stopping_rounds.
2026-05-28 20:50:46 | INFO | centaurdrug.admet.prototype | Best iteration: 298
2026-05-28 20:50:46 | INFO | centaurdrug.admet.prototype | Best score: 1.1451658241603238


## 12. Evaluate on train, validation, and test

In [14]:
train_pred = final_model.predict(X_train_scaled)
early_pred = final_model.predict(X_early_scaled)
valid_pred = final_model.predict(X_valid_scaled)
test_pred = final_model.predict(X_test_scaled)

metrics_report = {
    "train_core": regression_metrics(y_train, train_pred),
    "early_stop": regression_metrics(y_early, early_pred),
    "validation": regression_metrics(y_valid, valid_pred),
    "test": regression_metrics(y_test, test_pred),
}

for split_name, metrics in metrics_report.items():
    print_metrics(split_name, metrics)

metrics_path = ARTIFACT_DIR / "metrics_report.json"
metrics_path.write_text(json.dumps(metrics_report, indent=2), encoding="utf-8")
logger.info("Saved metrics report: %s", metrics_path)

pd.DataFrame(metrics_report).T

2026-05-28 20:50:54 | INFO | centaurdrug.admet.prototype | Saved metrics report: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/metrics_report.json


train_core
  mae: 0.44811
  mse: 0.39659
  rmse: 0.62975
  r2: 0.91957
early_stop
  mae: 0.80521
  mse: 1.31140
  rmse: 1.14517
  r2: 0.78775
validation
  mae: 0.74336
  mse: 1.10505
  rmse: 1.05121
  r2: 0.80172
test
  mae: 0.66011
  mse: 0.92298
  rmse: 0.96072
  r2: 0.88821


,mae,mse,rmse,r2
train_core,0.448115,0.396590,0.629754,0.919574
early_stop,0.805205,1.311405,1.145166,0.787750
validation,0.743365,1.105048,1.051213,0.801721
test,0.660110,0.922979,0.960718,0.888205


## 13. Error analysis

In [15]:
error_df = test_df[[SMILES_COL, TARGET_COL, "scaffold"]].copy()
error_df["prediction"] = test_pred
error_df["absolute_error"] = np.abs(error_df[TARGET_COL] - error_df["prediction"])
error_df = error_df.sort_values("absolute_error", ascending=False)

error_path = ARTIFACT_DIR / "test_error_analysis.csv"
error_df.to_csv(error_path, index=False)
logger.info("Saved error analysis: %s", error_path)

display(error_df.head(20))

2026-05-28 20:51:55 | INFO | centaurdrug.admet.prototype | Saved error analysis: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/test_error_analysis.csv


,Drug,Y,scaffold,prediction,absolute_error
615,[La],0.947184,acyclic::[La],-6.276494,7.223678
321,Clc1c(Cl)c(Cl)c(Oc2c(Cl)c(Cl)c(Cl)c(Cl)c2Cl)c(...,-12.950000,c1ccc(Oc2ccccc2)cc1,-8.346273,4.603727
215,[O]=[Co][OH],-6.275971,acyclic::[O]=[Co][OH],-2.170531,4.105440
647,CC(=O)C(N=Nc1ccc(-c2ccc(N=NC(C(C)=O)C(=O)NN)c(...,-8.705297,c1ccc(-c2ccccc2)cc1,-4.606363,4.098934
644,ClCc1ccc(-c2ccc(CCl)cc2)cc1,-10.098914,c1ccc(-c2ccccc2)cc1,-6.405477,3.693437
357,COC(=O)c1c(Cl)c(Cl)c(Cl)c(Cl)c1C#N.C[O-].Cc1cc...,-7.532415,c1ccc(N=Nc2ccccc2)cc1.c1ccccc1,-3.972531,3.559884
249,CCCCC(CC)COC(=O)OOC(=O)OCC(CC)CCCC,-2.595175,acyclic::CCCCC(CC)COC(=O)OOC(=O)OCC(CC)CCCC,-6.100409,3.505234
256,[Be],-7.255851,acyclic::[Be],-3.789048,3.466803
366,CCCCNC1CC(C)(C)NC(C)(C)C1,0.574615,C1CCNCC1,-2.869696,3.444311
98,CC[N+](CC)(CC)CC.O=S(=O)([O-])C(F)(F)C(F)(F)C(...,-0.072181,acyclic::CC[N+](CC)(CC)CC.O=S(=O)([O-])C(F)(F)...,-3.210605,3.138424


## 14. Save production-style artifacts

We save more than the model:

```text
model.joblib
feature_extractor.joblib
scaler.joblib
training_metadata.json
metrics_report.json
cv_hyperparameter_results.csv
split_report.csv
training_rejections.csv
test_error_analysis.csv
```

This is a minimal reproducible artifact package.

In [16]:
model_path = ARTIFACT_DIR / "xgboost_aqsol_model.joblib"
feature_extractor_path = ARTIFACT_DIR / "feature_extractor.joblib"
scaler_path = ARTIFACT_DIR / "scaler.joblib"

joblib.dump(final_model, model_path)
joblib.dump(feature_extractor, feature_extractor_path)
joblib.dump(scaler, scaler_path)

metadata = {
    "project": "CentaurDrug",
    "dataset": DATASET_NAME,
    "task_type": "regression",
    "target": "aqueous_solubility",
    "smiles_col": SMILES_COL,
    "target_col": TARGET_COL,
    "model_type": "XGBRegressor",
    "features": {
        "morgan_radius": feature_extractor.radius,
        "morgan_n_bits": feature_extractor.n_bits,
        "maccs_bits": 167,
        "rdkit_descriptors": feature_extractor.descriptor_names_,
    },
    "split_strategy": "Bemis-Murcko scaffold grouped split + GroupKFold scaffold CV",
    "best_params": best_params,
    "random_seed": RANDOM_SEED,
    "metrics": metrics_report,
    "rejection_contract": {
        "invalid_input_status": "rejected",
        "possible_reasons": ["missing_smiles", "empty_smiles", "invalid_smiles"],
    },
}

metadata_path = ARTIFACT_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

logger.info("Saved model: %s", model_path)
logger.info("Saved feature extractor: %s", feature_extractor_path)
logger.info("Saved scaler: %s", scaler_path)
logger.info("Saved metadata: %s", metadata_path)

2026-05-28 20:52:01 | INFO | centaurdrug.admet.prototype | Saved model: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/xgboost_aqsol_model.joblib
2026-05-28 20:52:01 | INFO | centaurdrug.admet.prototype | Saved feature extractor: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/feature_extractor.joblib
2026-05-28 20:52:01 | INFO | centaurdrug.admet.prototype | Saved scaler: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/scaler.joblib
2026-05-28 20:52:01 | INFO | centaurdrug.admet.prototype | Saved metadata: /home/aliaqil/Desktop/projects/centaurdrug/prototypes/models/admet_aqsol_xgboost_prototype/training_metadata.json


## 15. Optional MLflow logging

If MLflow is available, this logs:

- parameters
- metrics
- artifacts

Later, in production, this becomes part of `src/models/registry.py`.

In [17]:
if USE_MLFLOW:
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run(run_name="aqsol-xgboost-scaffold-cv"):
        mlflow.log_params(best_params)
        mlflow.log_param("dataset", DATASET_NAME)
        mlflow.log_param("feature_set", "morgan+maccs+rdkit_descriptors")
        mlflow.log_param("split_strategy", "scaffold_grouped")
        mlflow.log_param("random_seed", RANDOM_SEED)

        for split_name, metrics in metrics_report.items():
            for metric_name, value in metrics.items():
                mlflow.log_metric(f"{split_name}_{metric_name}", value)

        mlflow.log_artifact(str(model_path))
        mlflow.log_artifact(str(feature_extractor_path))
        mlflow.log_artifact(str(scaler_path))
        mlflow.log_artifact(str(metadata_path))
        mlflow.log_artifact(str(metrics_path))
        mlflow.log_artifact(str(cv_results_path))
        mlflow.log_artifact(str(error_path))
        mlflow.log_artifact(str(rejection_path))

    logger.info("Logged run to MLflow experiment: %s", MLFLOW_EXPERIMENT_NAME)
else:
    logger.warning("MLflow is not available. Skipping MLflow logging.")

2026/05/28 20:52:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/28 20:52:08 INFO mlflow.store.db.utils: Updating database tables
2026/05/28 20:52:14 INFO mlflow.tracking.fluent: Experiment with name 'centaurdrug-admet-aqsol-xgboost' does not exist. Creating a new experiment.
2026-05-28 20:52:15 | INFO | centaurdrug.admet.prototype | Logged run to MLflow experiment: centaurdrug-admet-aqsol-xgboost


## 16. Production-style inference contract

This is the function that later becomes your FastAPI endpoint logic.

Input:

```json
{ "smiles": "CCO" }
```

Valid output:

```json
{
  "status": "ok",
  "canonical_smiles": "CCO",
  "prediction": -0.42,
  "model": "xgboost_aqsol"
}
```

Invalid output:

```json
{
  "status": "rejected",
  "reason": "invalid_smiles"
}
```

In [18]:
loaded_model = joblib.load(model_path)
loaded_feature_extractor = joblib.load(feature_extractor_path)
loaded_scaler = joblib.load(scaler_path)


def predict_solubility(smiles: Any) -> Dict[str, Any]:
    validation = validate_smiles(smiles)

    if not validation.is_valid:
        logger.info("Rejected molecule: %s | reason=%s", smiles, validation.rejection_reason)
        return {
            "status": "rejected",
            "reason": validation.rejection_reason,
            "original_smiles": str(smiles),
        }

    X = loaded_feature_extractor.transform([validation.canonical_smiles])
    X_scaled = loaded_scaler.transform(X)
    pred = float(loaded_model.predict(X_scaled)[0])

    return {
        "status": "ok",
        "original_smiles": validation.original_smiles,
        "canonical_smiles": validation.canonical_smiles,
        "prediction": pred,
        "prediction_unit": "TDC_AqSolDB_target_scale",
        "model": "xgboost_aqsol",
    }


examples = [
    "CCO",
    "CC(=O)Oc1ccccc1C(=O)O",
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "not_a_smiles",
    "",
]

pd.DataFrame([predict_solubility(s) for s in examples])

2026-05-28 20:52:19 | INFO | centaurdrug.admet.prototype | Rejected molecule: not_a_smiles | reason=invalid_smiles
2026-05-28 20:52:19 | INFO | centaurdrug.admet.prototype | Rejected molecule:  | reason=empty_smiles


,status,original_smiles,canonical_smiles,prediction,prediction_unit,model,reason
0,ok,CCO,CCO,0.885075,TDC_AqSolDB_target_scale,xgboost_aqsol,NaN
1,ok,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,-1.916345,TDC_AqSolDB_target_scale,xgboost_aqsol,NaN
2,ok,Cn1cnc2c1c(=O)n(C)c(=O)n2C,Cn1c(=O)c2c(ncn2C)n(C)c1=O,-1.150848,TDC_AqSolDB_target_scale,xgboost_aqsol,NaN
3,rejected,not_a_smiles,NaN,NaN,NaN,NaN,invalid_smiles
4,rejected,,NaN,NaN,NaN,NaN,empty_smiles


## 17. FastAPI production sketch

Do not run this cell inside the notebook as a server. This is the exact logic we later move into:

```text
src/api/main.py
src/models/predict.py
```


In [ ]:
FASTAPI_SKETCH = r'''
from pydantic import BaseModel
from fastapi import FastAPI

app = FastAPI(title="CentaurDrug ADMET API")

class SolubilityRequest(BaseModel):
    smiles: str

@app.post("/admet/solubility")
def predict_aqsol(request: SolubilityRequest):
    return predict_solubility(request.smiles)
'''

print(FASTAPI_SKETCH)

# Next step after this notebook works

Convert into production modules:

```text
src/models/validation.py
src/models/datasets.py
src/models/splitting.py
src/models/featurizers.py
src/models/metrics.py
src/models/train_aqsol_xgboost.py
src/models/predict.py
src/models/registry.py
```

Then wire it with:

```text
configs/training.yaml
dvc.yaml
dags/centaurdrug_training_pipeline.py
src/api/main.py
Prometheus metrics
Grafana dashboard
```

This notebook is now a real bridge between experimentation and production.